# 第 21 章互动实验：用自己的光栅器渲染立方体

调用你实现的 look_at、perspective、to_screen、render_depth，渲染深度图，再加上逐面 Lambert 光照。

In [ ]:
%matplotlib inline
import importlib.util, os
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["PingFang SC", "Heiti SC", "Noto Sans CJK SC", "SimHei", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False

CHAPTER_NAME = "21_3D数学与渲染管线"
ASSIGNMENT = "21-1_软光栅.py"

def find_chapter(name):
    here = Path.cwd().resolve()
    for p in [here, *here.parents]:
        if p.name == name:
            return p
        if (p / name).is_dir():
            return p / name
    raise FileNotFoundError("请在课程目录内启动 Jupyter")

CHAPTER = find_chapter(CHAPTER_NAME)
COURSE_ROOT = CHAPTER.parent

def load_module(chapter_dir, filename):
    override = os.environ.get("COURSE_IMPL_DIR")
    base = Path(override) / chapter_dir.name if override else chapter_dir
    spec = importlib.util.spec_from_file_location(Path(filename).stem, base / filename)
    mod = importlib.util.module_from_spec(spec)
    spec.loader.exec_module(mod)
    return mod

mod = load_module(CHAPTER, ASSIGNMENT)

def attempt(fn, *args, **kwargs):
    """调用作业函数；还没实现时打印提示并返回 None，而不是中断整个 Notebook。"""
    try:
        return fn(*args, **kwargs)
    except NotImplementedError as exc:
        print(f"先完成作业里的 {fn.__name__}：{exc}")
        return None

try:
    import cv2  # 可选，用于和 OpenCV 对照
except ImportError:
    cv2 = None
print("作业文件：", CHAPTER / ASSIGNMENT, "｜OpenCV：", "可用" if cv2 else "未安装（对照实验会跳过）")


In [ ]:
V = np.array([[-1, -1, -1], [1, -1, -1], [1, 1, -1], [-1, 1, -1],
              [-1, -1, 1], [1, -1, 1], [1, 1, 1], [-1, 1, 1]], float)
FACES = [(0, 1, 2), (0, 2, 3), (4, 6, 5), (4, 7, 6), (0, 4, 5), (0, 5, 1),
         (3, 2, 6), (3, 6, 7), (0, 3, 7), (0, 7, 4), (1, 5, 6), (1, 6, 2)]
W, H = 240, 180

def render(eye, fov=50.0):
    view = mod.look_at(np.array(eye, float), np.zeros(3), np.array([0, 1, 0.0]))
    proj = mod.perspective(fov, W / H, 0.5, 20.0)
    s = mod.to_screen(V, proj @ view, W, H)
    tris = np.stack([s[list(f)] for f in FACES])
    return tris, mod.render_depth(tris, W, H)

out = attempt(render, [3.0, 2.5, 5.0])
if out is not None:
    tris, depth = out
    plt.imshow(np.where(np.isfinite(depth), depth, np.nan), cmap="viridis"); plt.colorbar(label="深度 [0,1]")
    plt.title("z-buffer 深度图"); plt.show()

## Lambert 光照

逐个三角形单独光栅化，再用深度最小者决定每个像素属于哪个面，乘上 max(0, n·l)。

In [ ]:
if out is not None:
    light = np.array([0.4, 0.8, 0.6]); light /= np.linalg.norm(light)
    per_face = np.stack([mod.render_depth(tris[i:i + 1], W, H) for i in range(len(FACES))])
    face_id = np.argmin(per_face, axis=0); covered = np.isfinite(per_face.min(axis=0))
    shade = np.zeros((H, W))
    for i, (a, b, c) in enumerate(FACES):
        n = np.cross(V[b] - V[a], V[c] - V[a]); n /= np.linalg.norm(n)
        shade[(face_id == i) & covered] = 0.15 + 0.85 * max(0.0, float(n @ light))
    plt.imshow(shade, cmap="gray", vmin=0, vmax=1); plt.title("逐面 Lambert"); plt.show()

## 实验：near 平面与深度精度

In [ ]:
if out is not None:
    z = np.linspace(-0.5, -20, 400)
    for near in (0.01, 0.5, 2.0):
        P = mod.perspective(50, 1.0, near, 20.0)
        d = mod.to_screen(np.stack([np.zeros_like(z), np.zeros_like(z), z], 1), P, 2, 2)[:, 2]
        plt.plot(-z, d, label=f"near={near}")
    plt.xlabel("到相机的距离"); plt.ylabel("屏幕深度"); plt.legend(); plt.title("深度是非线性的"); plt.show()

## 破坏实验与记录

- 把 up 改成 (0, -1, 0)，图像怎样？
- 把相机放到立方体内部，会发生什么？`to_screen` 的 NaN 处理在哪里起作用？
- 把 OpenGL 和 OpenCV 相机约定的对照写进 `实验记录.md`。